This notebook was originally going to be a demo of how to impose an effect of a given roughness ... though it highlights a problem with the current implementation: imposing roughness (see `sigma_gain` output in `compute_offset()`)  isn't accounted for in minimizing the norm of the offsets.  In other words, imposing a roughness has more dramatic impacts on the images than necessary.  To fix this we should include the space-varying-offsets associated with `sigma_gain` in our minimization objective.

In [ ]:
import matplotlib.pyplot as plt
import hglm
from hglm.demo.hglm_image_demo import *

seed = 0
shape = 5, 5
a, b, num_img = 2, 1, 3

rng = np.random.default_rng(seed=seed)
y = rng.standard_normal(size=(b, num_img, np.prod(shape)))

mask_idx = hglm.mask.get_mask_idx(np.ones(shape))
exp = Experiment(y=y, mask_idx=mask_idx, x=np.arange(num_img)[np.newaxis, :], contrast=[True], add_bias=True)

In [ ]:
def plot_img_row(exp, vmin, vmax):
    plt.figure()
    fig, ax = plt.subplots(1, num_img + 1)
    fig.set_size_inches(10, 3)
    for idx in range(num_img):
        plt.sca(ax[idx])
        y = exp.y[:, idx, :].reshape(shape)
        plt.imshow(y, cmap='gray', vmin=vmin, vmax=vmax)
        plt.gca().set_title(f'Image{idx} (x={idx})')
        plt.axis('off')
    
    plt.sca(ax[-1])
    ax[-1]
    _y = exp.y[0, :, :]
    x = exp.x[1, :]
    _x = np.tile(x[:, np.newaxis], (1, _y.shape[1]))
    plt.scatter(_x, _y)
    plt.plot(x, (exp.y.mean(axis=2) @ exp.h[1]).flatten())
    plt.ylim(vmin, vmax)

In [ ]:
vmin=-2.5
vmax=2.5
plot_img_row(exp, vmin=vmin, vmax=vmax)
for rough, p_val in ((None, .1),
                     (120, .1)):
    exp_w_effect, effect, rough = exp.impose_effect(seed=seed, mask=exp.mask_idx > -1, p_val=p_val, rough=rough)
    plot_img_row(exp_w_effect, vmin=vmin, vmax=vmax)
    print(rough)